# NB30 · Networking, HTTP & APIs

`advanced/` · notebook 30 of 32 · 24 questions + a mini-project

Almost every real program talks to another one over a network: a weather
API, a payment service, a database, a browser. This notebook shows you
what is happening underneath: raw **sockets** sending bytes, **HTTP**
(methods, status codes, headers, JSON bodies) layered on top, and the
standard-library tools that speak it: `urllib.parse` to build URLs,
`urllib.request` to make requests, and `http.server` to answer them.

**Everything runs on your own machine.** Each server listens on
`127.0.0.1` (your computer only, never the internet) on port `0`, which
tells the operating system to pick any free port. Servers run in
background threads (NB27), every request has a timeout, and every server
is shut down at the end of its cell. Port numbers change every run, so
nothing prints them.

**Why the standard library?** In real projects you'll usually install
`requests` or `httpx`. They are friendlier wrappers around exactly what
you practise here, and Q24 shows what they add. Knowing the layer below
is what lets you debug them when a request misbehaves.

Each server's handler defines `log_message` to do nothing. Without it,
`http.server` writes a log line for every request to stderr.

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB27: threads and asyncio
- [ ] NB13: JSON

## What you'll practise

- How HTTP works: methods, status codes, headers, JSON bodies, REST conventions
- Sockets: a tiny echo server and client on localhost in a thread
- http.server: serving a small JSON API inside the notebook
- urllib.request and urllib.parse: query strings, POST bodies, errors
- Timeouts, retries with backoff, handling 4xx vs 5xx
- What requests and httpx add (explained; exercises stay standard-library)

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L1 · Extremely Easy

### NB30-Q01 · L1 · URLs · Write

Split `url` into its parts with `urllib.parse.urlsplit` and print the
scheme, the host (`netloc`), the path and the query string, one per
line.

```text
https
shop.example.com
/api/orders
status=paid&page=2
```

**Starting code (already in the cell below):**

```python
from urllib.parse import urlsplit

url = "https://shop.example.com/api/orders?status=paid&page=2"
```

<details><summary><b>Hint</b></summary>

`urlsplit(url)` returns an object with attributes `.scheme`, `.netloc`, `.path` and `.query`.

</details>

<details><summary><b>Expected output</b></summary>

```text
https
shop.example.com
/api/orders
status=paid&page=2
```

</details>

<details><summary><b>Solution</b></summary>

```python
parts = urlsplit(url)
print(parts.scheme)
print(parts.netloc)
print(parts.path)
print(parts.query)
```

A URL packs several answers into one string: the *scheme* (which protocol), the *host* (which machine), the *path* (which resource on it) and the *query* (extra parameters after `?`). `urlsplit` cuts it at the right places, so you never slice URLs by hand.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
parts = urlsplit(url)
print(*parts[:4], sep="\n")
```

The result is a named tuple (NB15), so it also unpacks and slices like a tuple. Seniors parse URLs with `urlsplit` rather than `split("/")`, which breaks on ports, user names and `?` inside paths.

</details>

In [ ]:
from urllib.parse import urlsplit

url = "https://shop.example.com/api/orders?status=paid&page=2"

# your code here


### NB30-Q02 · L1 · HTTP status codes · Predict

`http.HTTPStatus` names every status code. Predict the output.

**What does this print?**

```python
from http import HTTPStatus

print(HTTPStatus.NOT_FOUND.value, HTTPStatus(201).phrase)
print(HTTPStatus(503).is_server_error, HTTPStatus(404).is_server_error)
```

<details><summary><b>Hint</b></summary>

2xx means success, 4xx means the *client* made a mistake, 5xx means the *server* failed.

</details>

<details><summary><b>Expected output</b></summary>

```text
404 Created
True False
```

</details>

<details><summary><b>Solution</b></summary>

`NOT_FOUND` is `404`. `201` is `Created`, the usual answer to a successful POST. `503` (Service Unavailable) is in the 5xx range, so it is a server error; `404` is a client error. The first digit is the category, and it tells you what to do next: fix your request (4xx) or try again later (5xx).

</details>

<details><summary><b>Senior dev solution</b></summary>

Seniors write `HTTPStatus.NOT_FOUND` instead of a bare `404` in server code: the name says what it means, and a typo becomes an `AttributeError` instead of a wrong number.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## L2 · Easy

### NB30-Q03 · L2 · Query strings · Write

Build the full search URL from `base` and `params` with
`urllib.parse.urlencode`, and print it.

```text
https://shop.example.com/search?q=red+shoes&size=42&sort=price
```

**Starting code (already in the cell below):**

```python
from urllib.parse import urlencode

base = "https://shop.example.com/search"
params = {"q": "red shoes", "size": 42, "sort": "price"}
```

<details><summary><b>Hint</b></summary>

`urlencode(params)` turns the dict into `key=value&key=value`. Join it to `base` with a `?`.

</details>

<details><summary><b>Expected output</b></summary>

```text
https://shop.example.com/search?q=red+shoes&size=42&sort=price
```

</details>

<details><summary><b>Solution</b></summary>

```python
print(base + "?" + urlencode(params))
```

`urlencode` converts each value to text, escapes anything unsafe (the space in `red shoes` becomes `+`), and joins the pairs with `&`. Doing it by hand works until a value contains `&`, `=` or `#`, which then silently changes the meaning of the URL (Q14).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
url = f"{base}?{urlencode(params)}"
print(url)
```

Seniors keep the URL in a named variable so the same value can be logged and requested. With `requests` (Q24) you'd pass `params=params` and the library calls `urlencode` for you.

</details>

In [ ]:
from urllib.parse import urlencode

base = "https://shop.example.com/search"
params = {"q": "red shoes", "size": 42, "sort": "price"}

# your code here


### NB30-Q04 · L2 · Query strings · Predict

Three ways to escape the same text. They don't all agree. Predict the
output.

**What does this print?**

```python
from urllib.parse import quote, quote_plus, urlencode

term = "fish & chips"
print(quote(term))
print(quote_plus(term))
print(urlencode({"q": term}))
```

<details><summary><b>Hint</b></summary>

`quote` is for *paths*, where a space becomes `%20`. Query strings traditionally write a space as `+`.

</details>

<details><summary><b>Expected output</b></summary>

```text
fish%20%26%20chips
fish+%26+chips
q=fish+%26+chips
```

</details>

<details><summary><b>Solution</b></summary>

`quote` escapes for a URL **path**: the space becomes `%20` and `&` becomes `%26`. `quote_plus` escapes for a **query string** value, where a space is `+`. `urlencode` builds a whole query string: it uses `quote_plus` on each key and value and adds the `q=` and the `&` separators.

</details>

<details><summary><b>Senior dev solution</b></summary>

The rule seniors follow: `quote` for a path segment (`/files/fish%20%26%20chips`), `urlencode` for the part after `?`. Mixing them up gives URLs that look right but are read differently by the server.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB30-Q05 · L2 · Query strings · Write

Read the query string of `url` with `urllib.parse.parse_qs`. Print the
list of tags, then the page number **as an int**.

```text
tags: ['sale', 'new']
page: 3
```

**Starting code (already in the cell below):**

```python
from urllib.parse import parse_qs, urlsplit

url = "https://shop.example.com/search?tag=sale&tag=new&page=3"
```

<details><summary><b>Hint</b></summary>

`parse_qs` maps each key to a **list** of values, because a key can repeat. Everything in a URL is text.

</details>

<details><summary><b>Expected output</b></summary>

```text
tags: ['sale', 'new']
page: 3
```

</details>

<details><summary><b>Solution</b></summary>

```python
query = parse_qs(urlsplit(url).query)
print("tags:", query["tag"])
print("page:", int(query["page"][0]))
```

`parse_qs` returns `{'tag': ['sale', 'new'], 'page': ['3']}`. Every value is a list (a key may appear several times) and every value is a string (a URL is text), so the page needs `[0]` and then `int()`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
query = parse_qs(urlsplit(url).query)
page = int(query.get("page", ["1"])[0])
print("tags:", query.get("tag", []))
print("page:", page)
```

Query strings come from outside, so seniors give every parameter a default with `.get` and convert it once. A missing `page` then means page 1 instead of a `KeyError` and a 500 response.

</details>

In [ ]:
from urllib.parse import parse_qs, urlsplit

url = "https://shop.example.com/search?tag=sale&tag=new&page=3"

# your code here


### NB30-Q06 · L2 · HTTP requests · Write

Build (but don't send) a `urllib.request.Request` that POSTs `order` as
JSON to `url`: the body is the JSON text encoded to bytes, with a
`Content-Type` of `application/json`. Print the method, the
content type (read it back with `get_header("Content-type")`) and the
body decoded.

```text
POST
application/json
{"item": "apple", "qty": 3}
```

**Starting code (already in the cell below):**

```python
import json
import urllib.request

url = "http://127.0.0.1/api/orders"
order = {"item": "apple", "qty": 3}
```

<details><summary><b>Hint</b></summary>

`json.dumps(order).encode()` gives the bytes. `Request(url, data=..., headers={...})`. `urllib` stores header names with only the first letter capitalised.

</details>

<details><summary><b>Expected output</b></summary>

```text
POST
application/json
{"item": "apple", "qty": 3}
```

</details>

<details><summary><b>Solution</b></summary>

```python
body = json.dumps(order).encode()
request = urllib.request.Request(url, data=body,
                                 headers={"Content-Type": "application/json"})
print(request.get_method())
print(request.get_header("Content-type"))
print(request.data.decode())
```

An HTTP request is a **method** (what to do), a URL (to what), **headers** (facts about the request) and an optional **body**. A body travels as bytes, so the JSON text is encoded. The `Content-Type` header tells the server how to read those bytes. Giving `data` switches the method from GET to POST automatically.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
body = json.dumps(order).encode()
request = urllib.request.Request(url, data=body, method="POST",
                                 headers={"Content-Type": "application/json"})
print(request.get_method())
print(request.get_header("Content-type"))
print(request.data.decode())
```

Seniors pass `method="POST"` explicitly rather than relying on the implicit switch, so the code reads the same as the API docs. (`urllib` stores `Content-Type` as `Content-type`; HTTP header names are case-insensitive, but this lookup is not.)

</details>

In [ ]:
import json
import urllib.request

url = "http://127.0.0.1/api/orders"
order = {"item": "apple", "qty": 3}

# your code here


### NB30-Q07 · L2 · HTTP requests · Error

A request to a local API. Which error does this raise?

**Which error does this raise?**

```python
import urllib.request

urllib.request.urlopen("localhost/api/orders", timeout=2)
```

<details><summary><b>Hint</b></summary>

Look at the start of the URL. How would `urllib` know which protocol to speak?

</details>

<details><summary><b>Expected output</b></summary>

```text
ValueError
```

</details>

<details><summary><b>Solution</b></summary>

`ValueError`: the URL has no scheme (`http://`), so `urllib` can't tell which protocol to use and rejects it before connecting. The fix is `"http://localhost/api/orders"`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from urllib.parse import urlsplit

url = "localhost/api/orders"
print(urlsplit(url).scheme in ("http", "https"))
```

When a URL comes from config or a user, seniors validate it up front with `urlsplit` and fail with a clear message, instead of letting a confusing error surface deep inside a request.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

---

## L3 · Intermediate

### NB30-Q08 · L3 · Sockets · Write

An echo server is already listening in a background thread: it reads
one message and sends it back in capitals. Write the client: connect
with `socket.create_connection(listener.getsockname(), timeout=5)`,
send `b"hello sockets"`, receive the reply, and print it decoded. Then
wait for the server thread and close the listener.

```text
HELLO SOCKETS
```

**Starting code (already in the cell below):**

```python
import socket
import threading

def echo_once(listener):
    conn, _ = listener.accept()
    with conn:
        data = conn.recv(1024)
        conn.sendall(data.upper())

listener = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
listener.bind(("127.0.0.1", 0))
listener.listen()
listener.settimeout(5)
server_thread = threading.Thread(target=echo_once, args=(listener,), daemon=True)
server_thread.start()
```

<details><summary><b>Hint</b></summary>

Use the client socket in a `with` block. `sendall(bytes)` sends, `recv(1024)` receives up to 1024 bytes. Sockets carry **bytes**, so decode the reply.

</details>

<details><summary><b>Expected output</b></summary>

```text
HELLO SOCKETS
```

</details>

<details><summary><b>Solution</b></summary>

```python
with socket.create_connection(listener.getsockname(), timeout=5) as client:
    client.sendall(b"hello sockets")
    reply = client.recv(1024)
print(reply.decode())
server_thread.join(timeout=5)
listener.close()
```

A TCP socket is a two-way pipe of bytes. The server *binds* an address, *listens*, and *accepts* a connection; the client *connects*. Then both sides `sendall` and `recv` bytes. Everything above this (HTTP, databases, email) is an agreement about what those bytes mean. The timeouts mean a mistake gives an error instead of a frozen notebook.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
with socket.create_connection(listener.getsockname(), timeout=5) as client:
    client.sendall("hello sockets".encode("utf-8"))
    reply = client.recv(1024)
print(reply.decode("utf-8"))
server_thread.join(timeout=5)
listener.close()
```

Seniors name the encoding on both sides: the network has no idea what text is. One `recv` is fine for a tiny message on your own machine, but TCP may split data anywhere, so real protocols mark where a message ends (Q18).

</details>

In [ ]:
import socket
import threading

def echo_once(listener):
    conn, _ = listener.accept()
    with conn:
        data = conn.recv(1024)
        conn.sendall(data.upper())

listener = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
listener.bind(("127.0.0.1", 0))
listener.listen()
listener.settimeout(5)
server_thread = threading.Thread(target=echo_once, args=(listener,), daemon=True)
server_thread.start()

# your code here


### NB30-Q09 · L3 · http.server · Write

Write a `Handler` (a subclass of `BaseHTTPRequestHandler`) whose
`do_GET` answers `/health` with status `200` and the JSON
`{"status": "ok", "items": 3}`, and anything else with `404` and
`{"error": "not found"}`. Use the given `send_json` (copy it into your
class) and the `start_server` helper. Fetch `/health`, print the status
and the decoded JSON, then shut the server down.

```text
200 {'status': 'ok', 'items': 3}
```

**Starting code (already in the cell below):**

```python
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

# Paste this into your Handler class:
def send_json(self, status, data):
    body = json.dumps(data).encode()
    self.send_response(status)
    self.send_header("Content-Type", "application/json")
    self.send_header("Content-Length", str(len(body)))
    self.end_headers()
    self.wfile.write(body)
```

<details><summary><b>Hint</b></summary>

`self.path` holds the requested path. Add `log_message(self, format, *args)` that does nothing. On the client, `json.load(response)` reads and parses the body; `response.status` is the code.

</details>

<details><summary><b>Expected output</b></summary>

```text
200 {'status': 'ok', 'items': 3}
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/health":
            self.send_json(200, {"status": "ok", "items": 3})
        else:
            self.send_json(404, {"error": "not found"})

    send_json = send_json

    def log_message(self, format, *args):
        pass

server, base = start_server(Handler)
with urllib.request.urlopen(base + "/health", timeout=5) as response:
    print(response.status, json.load(response))
server.shutdown()
server.server_close()
```

`http.server` parses each request and calls the method named after the HTTP verb: `do_GET`, `do_POST` and so on. A response is a status line, headers, a blank line, then the body; `send_json` writes them in that order. `Content-Length` tells the client where the body ends. (`send_json = send_json` copies the given function into the class as a method; pasting its code in works the same.)

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from http import HTTPStatus

class Handler(BaseHTTPRequestHandler):
    routes = {"/health": lambda: (HTTPStatus.OK, {"status": "ok", "items": 3})}

    def do_GET(self):
        route = self.routes.get(self.path)
        status, data = route() if route else (HTTPStatus.NOT_FOUND, {"error": "not found"})
        self.send_json(status, data)

    send_json = send_json

    def log_message(self, format, *args):
        pass

server, base = start_server(Handler)
try:
    with urllib.request.urlopen(base + "/health", timeout=5) as response:
        print(response.status, json.load(response))
finally:
    server.shutdown()
    server.server_close()
```

A route table scales better than an `if` chain, and `try`/`finally` shuts the server down even if the request fails. `http.server` is for learning and tests only (its docs say not to use it in production); real services use a framework such as FastAPI or Flask.

</details>

In [ ]:
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

# Paste this into your Handler class:
def send_json(self, status, data):
    body = json.dumps(data).encode()
    self.send_response(status)
    self.send_header("Content-Type", "application/json")
    self.send_header("Content-Length", str(len(body)))
    self.end_headers()
    self.wfile.write(body)

# your code here


### NB30-Q10 · L3 · REST conventions · Predict

A REST API maps (method, path) pairs to actions. This function returns
the status code a conventional todo API would send. Predict the output.

**What does this print?**

```python
def route(method, path):
    if path == "/todos":
        if method == "GET":
            return 200
        if method == "POST":
            return 201
        return 405
    if path.startswith("/todos/"):
        if method in ("GET", "PUT"):
            return 200
        if method == "DELETE":
            return 204
        return 405
    return 404

print(route("POST", "/todos"), route("DELETE", "/todos/7"))
print(route("POST", "/todos/7"), route("GET", "/users"))
```

<details><summary><b>Hint</b></summary>

`/todos` is the collection, `/todos/7` is one item. 405 means "this path exists, but not for that method".

</details>

<details><summary><b>Expected output</b></summary>

```text
201 204
405 404
```

</details>

<details><summary><b>Solution</b></summary>

`201 204` then `405 404`. POST to a collection creates an item: `201 Created`. DELETE of one item succeeds with `204 No Content` (nothing to send back). POST to a single item isn't allowed: `405 Method Not Allowed`, because the path exists. `/users` doesn't exist at all: `404 Not Found`.

</details>

<details><summary><b>Senior dev solution</b></summary>

REST conventions in one line each: nouns in paths (`/todos/7`, not `/deleteTodo`), the method says the action, and the status says the outcome. Seniors keep to them so clients can guess an API's behaviour without reading every page of its docs.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB30-Q11 · L3 · HTTP requests · Write

The local server answers a POST to `/orders` with `201` and the order
it stored. Send `order` as a JSON body (method `POST`, header
`Content-Type: application/json`), print the status and the decoded
reply, then shut the server down.

```text
201 {'id': 1, 'item': 'apple', 'qty': 3}
```

**Starting code (already in the cell below):**

```python
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        data = json.loads(self.rfile.read(length))
        self.send_json(201, {"id": 1, **data})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)
order = {"item": "apple", "qty": 3}
```

<details><summary><b>Hint</b></summary>

Build a `urllib.request.Request` with `data=`, `method=` and `headers=` (as in Q06), then pass it to `urlopen` with a timeout.

</details>

<details><summary><b>Expected output</b></summary>

```text
201 {'id': 1, 'item': 'apple', 'qty': 3}
```

</details>

<details><summary><b>Solution</b></summary>

```python
body = json.dumps(order).encode()
request = urllib.request.Request(base + "/orders", data=body, method="POST",
                                 headers={"Content-Type": "application/json"})
with urllib.request.urlopen(request, timeout=5) as response:
    print(response.status, json.load(response))
server.shutdown()
server.server_close()
```

The server reads exactly `Content-Length` bytes of body, parses the JSON, and replies `201 Created` with the stored record, including the `id` it assigned. Echoing the created resource back is a REST convention: the client learns the new id without a second request.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def post_json(url, payload, timeout=5):
    request = urllib.request.Request(
        url, data=json.dumps(payload).encode(), method="POST",
        headers={"Content-Type": "application/json", "Accept": "application/json"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.status, json.load(response)

try:
    print(*post_json(base + "/orders", order))
finally:
    server.shutdown()
    server.server_close()
```

A small `post_json` helper means the headers and timeout are written once and can't be forgotten on the next call. The `Accept` header says which format you want back; well-behaved APIs honour it.

</details>

In [ ]:
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        data = json.loads(self.rfile.read(length))
        self.send_json(201, {"id": 1, **data})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)
order = {"item": "apple", "qty": 3}

# your code here


### NB30-Q12 · L3 · Query strings · Refactor

This builds a weather URL by gluing strings together. It works for
these values, but it is fragile. Rewrite it with `urlencode`.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
base = "https://api.example.com/weather"
city = "New York"
units = "metric"
url = base + "?city=" + city.replace(" ", "+") + "&units=" + units
print(url)
```

<details><summary><b>Hint</b></summary>

Put the parameters in a dict and let `urlencode` handle the escaping and the `&`.

</details>

<details><summary><b>Expected output</b></summary>

```text
https://api.example.com/weather?city=New+York&units=metric
```

</details>

<details><summary><b>Solution</b></summary>

```python
from urllib.parse import urlencode

base = "https://api.example.com/weather"
city = "New York"
units = "metric"
url = base + "?" + urlencode({"city": city, "units": units})
print(url)
```

The hand-made version only escapes spaces. A city like `Bath & Wells` or `São Paulo` would produce a broken or misread URL. `urlencode` escapes every value correctly and adds the separators, so the output is the same here and stays right for any input.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from urllib.parse import urlencode

BASE_URL = "https://api.example.com/weather"
params = {"city": "New York", "units": "metric"}
print(f"{BASE_URL}?{urlencode(params)}")
```

Keeping parameters in a dict means adding one is a one-line change, and the dict can be logged or tested on its own. Constant base URLs get `CAPITAL` names (PEP 8) and usually come from config (NB31).

</details>

In [ ]:
base = "https://api.example.com/weather"
city = "New York"
units = "metric"
url = base + "?city=" + city.replace(" ", "+") + "&units=" + units
print(url)


### NB30-Q13 · L3 · HTTP headers · Write

The local API wants the header `X-Api-Key: secret-123`. Make one
request to `/me` **with** the key and print the status, the response's
`Content-Type` header and the JSON. Then make one **without** it and
print the status code from the `HTTPError`. Shut the server down.

```text
200 application/json {'user': 'ada'}
401
```

**Starting code (already in the cell below):**

```python
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.headers.get("X-Api-Key") == "secret-123":
            self.send_json(200, {"user": "ada"})
        else:
            self.send_json(401, {"error": "missing or wrong key"})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)
```

<details><summary><b>Hint</b></summary>

Pass `headers={...}` to `Request`. Read a response header with `response.headers["Content-Type"]`. `urlopen` *raises* `urllib.error.HTTPError` for 4xx and 5xx; its `.code` is the status.

</details>

<details><summary><b>Expected output</b></summary>

```text
200 application/json {'user': 'ada'}
401
```

</details>

<details><summary><b>Solution</b></summary>

```python
request = urllib.request.Request(base + "/me", headers={"X-Api-Key": "secret-123"})
with urllib.request.urlopen(request, timeout=5) as response:
    print(response.status, response.headers["Content-Type"], json.load(response))
try:
    urllib.request.urlopen(base + "/me", timeout=5)
except urllib.error.HTTPError as err:
    print(err.code)
server.shutdown()
server.server_close()
```

Headers carry information *about* a request or response: who you are (`X-Api-Key`, `Authorization`), what format the body is in (`Content-Type`), how long it is (`Content-Length`). Without the key the server answers `401 Unauthorized`, and `urlopen` turns any 4xx or 5xx into an `HTTPError` exception rather than returning it.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
API_KEY = "secret-123"  # in real code: os.environ["API_KEY"] (NB31)

request = urllib.request.Request(base + "/me", headers={"X-Api-Key": API_KEY})
with urllib.request.urlopen(request, timeout=5) as response:
    print(response.status, response.headers.get_content_type(), json.load(response))
try:
    urllib.request.urlopen(base + "/me", timeout=5)
except urllib.error.HTTPError as err:
    print(err.code)
    err.close()
server.shutdown()
server.server_close()
```

Secrets never live in source code; they come from the environment. `get_content_type()` ignores extras like `; charset=utf-8`, which a raw string comparison would trip over. An `HTTPError` holds an open response, so seniors close it.

</details>

In [ ]:
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.headers.get("X-Api-Key") == "secret-123":
            self.send_json(200, {"user": "ada"})
        else:
            self.send_json(401, {"error": "missing or wrong key"})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)

# your code here


### NB30-Q14 · L3 · Query strings · Fix

The search term contains `&`, and the server ends up searching for
`"fish "` only. Fix how the URL is built so the parsed query becomes:

```text
{'q': ['fish & chips'], 'page': ['2']}
```

**Buggy code (copied into the cell below):**

```python
from urllib.parse import parse_qs, urlsplit

term = "fish & chips"
url = "https://shop.example.com/search?q=" + term + "&page=2"
print(parse_qs(urlsplit(url).query))
```

<details><summary><b>Hint</b></summary>

In a query string, `&` *separates* parameters. The `&` inside the value must be escaped.

</details>

<details><summary><b>Expected output</b></summary>

```text
{'q': ['fish & chips'], 'page': ['2']}
```

</details>

<details><summary><b>Solution</b></summary>

```python
from urllib.parse import parse_qs, urlencode, urlsplit

term = "fish & chips"
url = "https://shop.example.com/search?" + urlencode({"q": term, "page": 2})
print(parse_qs(urlsplit(url).query))
```

Glued in raw, the URL reads `q=fish & chips&page=2`. The server splits on every `&`, so it sees `q=fish `, a strange parameter ` chips` with no value (dropped), and `page=2`. `urlencode` escapes the value's `&` as `%26`, so it survives as part of `q`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from urllib.parse import parse_qs, urlencode, urlsplit, urlunsplit

term = "fish & chips"
query = urlencode({"q": term, "page": 2})
url = urlunsplit(("https", "shop.example.com", "/search", query, ""))
print(parse_qs(urlsplit(url).query))
```

`urlunsplit` is the mirror of `urlsplit`: it assembles a URL from its parts, so no piece is ever glued in by hand. This bug is also a security one: unescaped input can add parameters the server trusts.

</details>

In [ ]:
from urllib.parse import parse_qs, urlsplit

term = "fish & chips"
url = "https://shop.example.com/search?q=" + term + "&page=2"
print(parse_qs(urlsplit(url).query))


### NB30-Q15 · L3 · Retries and backoff · Write

Exponential backoff waits longer after each failure: `base_delay`,
then double, then double again, but never more than `max_delay`.
Print the list of waits for `attempts` retries.

```text
[0.5, 1.0, 2.0, 4.0, 4.0, 4.0]
```

**Starting code (already in the cell below):**

```python
base_delay = 0.5
max_delay = 4.0
attempts = 6
```

<details><summary><b>Hint</b></summary>

The wait before retry number `n` (counting from 0) is `base_delay * 2 ** n`, capped with `min()`.

</details>

<details><summary><b>Expected output</b></summary>

```text
[0.5, 1.0, 2.0, 4.0, 4.0, 4.0]
```

</details>

<details><summary><b>Solution</b></summary>

```python
waits = []
for n in range(attempts):
    waits.append(min(base_delay * 2 ** n, max_delay))
print(waits)
```

Doubling the wait gives a struggling server room to recover instead of being hammered by every client at once. The cap stops the waits growing absurdly long: after 20 failures, an uncapped wait would be days.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
waits = [min(base_delay * 2 ** n, max_delay) for n in range(attempts)]
print(waits)
```

In production, seniors add random *jitter* (for example `random.uniform(0, wait)`) so that a thousand clients that failed at the same moment don't all retry at the same moment too. Here it is left out so the output stays predictable.

</details>

In [ ]:
base_delay = 0.5
max_delay = 4.0
attempts = 6

# your code here


---

## L4 · Somewhat Difficult

### NB30-Q16 · L4 · 4xx vs 5xx · Write

Write `fetch_order(path)` that returns a message, and treat failures
**differently**: a 4xx means the request was wrong (don't retry), a 5xx
means the server failed (try again later). Print the message for each
path, then shut the server down.

```text
/orders/1: ok {'id': 1, 'total': 25.5}
/orders/99: client error 404, check the request (do not retry)
/orders/boom: server error 500, try again later
```

**Starting code (already in the cell below):**

```python
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/orders/1":
            self.send_json(200, {"id": 1, "total": 25.5})
        elif self.path == "/orders/boom":
            self.send_json(500, {"error": "database down"})
        else:
            self.send_json(404, {"error": "no such order"})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)
```

<details><summary><b>Hint</b></summary>

Catch `urllib.error.HTTPError` and look at `err.code`: `400 <= code < 500` is a client error, `500` and up is a server error.

</details>

<details><summary><b>Expected output</b></summary>

```text
/orders/1: ok {'id': 1, 'total': 25.5}
/orders/99: client error 404, check the request (do not retry)
/orders/boom: server error 500, try again later
```

</details>

<details><summary><b>Solution</b></summary>

```python
def fetch_order(path):
    try:
        with urllib.request.urlopen(base + path, timeout=5) as response:
            return f"ok {json.load(response)}"
    except urllib.error.HTTPError as err:
        if 400 <= err.code < 500:
            return f"client error {err.code}, check the request (do not retry)"
        return f"server error {err.code}, try again later"

for path in ["/orders/1", "/orders/99", "/orders/boom"]:
    print(f"{path}: {fetch_order(path)}")
server.shutdown()
server.server_close()
```

Not every non-200 means the same thing. A 404 will be a 404 every time, so retrying wastes time and load: report it and move on. A 500 (or 502, 503, 504) may be temporary, so a retry later can succeed. Treating every failure the same either hides real bugs or hammers a sick server.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from http import HTTPStatus

def fetch_order(path):
    try:
        with urllib.request.urlopen(base + path, timeout=5) as response:
            return f"ok {json.load(response)}"
    except urllib.error.HTTPError as err:
        with err:
            status = HTTPStatus(err.code)
        if status.is_client_error:
            return f"client error {err.code}, check the request (do not retry)"
        return f"server error {err.code}, try again later"

try:
    for path in ["/orders/1", "/orders/99", "/orders/boom"]:
        print(f"{path}: {fetch_order(path)}")
finally:
    server.shutdown()
    server.server_close()
```

`HTTPStatus(code).is_client_error` names the rule instead of encoding it as a range. Some 4xx codes *are* worth a retry: `408 Request Timeout` and `429 Too Many Requests` (wait for its `Retry-After` header). Seniors keep a short list of retryable codes rather than a blanket rule.

</details>

In [ ]:
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/orders/1":
            self.send_json(200, {"id": 1, "total": 25.5})
        elif self.path == "/orders/boom":
            self.send_json(500, {"error": "database down"})
        else:
            self.send_json(404, {"error": "no such order"})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)

# your code here


### NB30-Q17 · L4 · Timeouts · Error

This server accepts connections but never answers. Which error does
the request raise?

**Which error does this raise?**

```python
import socket
import urllib.request

with socket.socket() as silent:
    silent.bind(("127.0.0.1", 0))
    silent.listen()
    host, port = silent.getsockname()
    urllib.request.urlopen(f"http://{host}:{port}/", timeout=0.5)
```

<details><summary><b>Hint</b></summary>

The connection succeeds (the operating system queues it), but the reply never comes. What happens when the `timeout` runs out?

</details>

<details><summary><b>Expected output</b></summary>

```text
TimeoutError
```

</details>

<details><summary><b>Solution</b></summary>

`TimeoutError`: the request was sent, but no byte came back within 0.5 seconds. **Without** `timeout=`, `urlopen` would wait forever and freeze your program. The default is no timeout at all, which is why every network call in this notebook sets one.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import socket
import urllib.request

with socket.socket() as silent:
    silent.bind(("127.0.0.1", 0))
    silent.listen()
    host, port = silent.getsockname()
    try:
        urllib.request.urlopen(f"http://{host}:{port}/", timeout=0.5)
    except TimeoutError:
        print("no answer within 0.5s, giving up")
```

Seniors set a timeout on **every** network call and decide up front what a timeout means: retry, fall back to cached data, or fail with a clear message. `requests` also has no default timeout, which surprises many people.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB30-Q18 · L4 · Sockets · Fix

This server reads until the client has finished sending, then replies
in capitals. The client sends its message and waits for the reply, but
the two sides wait for each other until the client times out. Fix the
client so the output becomes:

```text
ORDER 42 SHIPPED
```

**Buggy code (copied into the cell below):**

```python
import socket
import threading

def shout_back(listener):
    conn, _ = listener.accept()
    with conn:
        conn.settimeout(3)
        chunks = []
        while True:
            chunk = conn.recv(1024)
            if not chunk:
                break
            chunks.append(chunk)
        conn.sendall(b"".join(chunks).upper())

listener = socket.create_server(("127.0.0.1", 0))
listener.settimeout(5)
server_thread = threading.Thread(target=shout_back, args=(listener,), daemon=True)
server_thread.start()

with socket.create_connection(listener.getsockname(), timeout=1) as client:
    client.sendall(b"order 42 shipped")
    reply = client.recv(1024)
print(reply.decode())
server_thread.join(timeout=5)
listener.close()
```

<details><summary><b>Hint</b></summary>

The server's loop stops when `recv` returns empty bytes, which only happens when the other side says it has finished *sending*. `client.shutdown(socket.SHUT_WR)` says exactly that, while still allowing the client to receive.

</details>

<details><summary><b>Expected output</b></summary>

```text
ORDER 42 SHIPPED
```

</details>

<details><summary><b>Solution</b></summary>

```python
import socket
import threading

def shout_back(listener):
    conn, _ = listener.accept()
    with conn:
        conn.settimeout(3)
        chunks = []
        while True:
            chunk = conn.recv(1024)
            if not chunk:
                break
            chunks.append(chunk)
        conn.sendall(b"".join(chunks).upper())

listener = socket.create_server(("127.0.0.1", 0))
listener.settimeout(5)
server_thread = threading.Thread(target=shout_back, args=(listener,), daemon=True)
server_thread.start()

with socket.create_connection(listener.getsockname(), timeout=1) as client:
    client.sendall(b"order 42 shipped")
    client.shutdown(socket.SHUT_WR)
    reply = client.recv(1024)
print(reply.decode())
server_thread.join(timeout=5)
listener.close()
```

TCP is a stream with no built-in message boundaries. The server can only know the message is complete when the client closes its sending side, but the client was waiting for a reply first: a deadlock, broken only by the client's timeout (`TimeoutError`). `shutdown(SHUT_WR)` sends the "I'm done sending" signal, the server's `recv` returns `b""`, and the reply flows back.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import socket
import threading

def shout_back(listener):
    conn, _ = listener.accept()
    with conn, conn.makefile("rb") as reader:
        conn.settimeout(3)
        line = reader.readline()
        conn.sendall(line.upper())

listener = socket.create_server(("127.0.0.1", 0))
listener.settimeout(5)
server_thread = threading.Thread(target=shout_back, args=(listener,), daemon=True)
server_thread.start()

with socket.create_connection(listener.getsockname(), timeout=1) as client:
    client.sendall(b"order 42 shipped\n")
    reply = client.recv(1024)
print(reply.decode().rstrip())
server_thread.join(timeout=5)
listener.close()
```

Real protocols *frame* their messages so the connection can stay open: one message per line (SMTP, Redis), or a length prefix (HTTP's `Content-Length`). The senior version ends each message with `\n` and reads one line, so the client can send again on the same connection.

</details>

In [ ]:
import socket
import threading

def shout_back(listener):
    conn, _ = listener.accept()
    with conn:
        conn.settimeout(3)
        chunks = []
        while True:
            chunk = conn.recv(1024)
            if not chunk:
                break
            chunks.append(chunk)
        conn.sendall(b"".join(chunks).upper())

listener = socket.create_server(("127.0.0.1", 0))
listener.settimeout(5)
server_thread = threading.Thread(target=shout_back, args=(listener,), daemon=True)
server_thread.start()

with socket.create_connection(listener.getsockname(), timeout=1) as client:
    client.sendall(b"order 42 shipped")
    reply = client.recv(1024)
print(reply.decode())
server_thread.join(timeout=5)
listener.close()


### NB30-Q19 · L4 · 4xx vs 5xx · Fix

The server clearly answers `404`, yet this code reports a network
problem. Fix the error handling so the output becomes:

```text
server answered 404
```

**Buggy code (copied into the cell below):**

```python
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_json(404, {"error": "not found"})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)

try:
    with urllib.request.urlopen(base + "/missing", timeout=5) as response:
        print(response.status)
except urllib.error.URLError:
    print("network problem: check your connection")
except urllib.error.HTTPError as err:
    print("server answered", err.code)
server.shutdown()
server.server_close()
```

<details><summary><b>Hint</b></summary>

`HTTPError` is a subclass of `URLError`. Python uses the **first** `except` clause that matches.

</details>

<details><summary><b>Expected output</b></summary>

```text
server answered 404
```

</details>

<details><summary><b>Solution</b></summary>

```python
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_json(404, {"error": "not found"})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)

try:
    with urllib.request.urlopen(base + "/missing", timeout=5) as response:
        print(response.status)
except urllib.error.HTTPError as err:
    print("server answered", err.code)
except urllib.error.URLError:
    print("network problem: check your connection")
server.shutdown()
server.server_close()
```

`except` clauses are tried top to bottom. Because every `HTTPError` *is* a `URLError`, the general clause caught the 404 first and the specific one never ran. Specific exceptions go first. The distinction matters: `HTTPError` means the server answered (look at the code), while a plain `URLError` means you never reached it (DNS, refused connection).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_json(404, {"error": "not found"})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)

try:
    with urllib.request.urlopen(base + "/missing", timeout=5) as response:
        print(response.status)
except urllib.error.HTTPError as err:
    with err:
        print("server answered", err.code)
except (urllib.error.URLError, TimeoutError) as err:
    print("network problem:", err)
finally:
    server.shutdown()
    server.server_close()
```

Seniors also catch `TimeoutError` alongside `URLError` (a slow read raises it directly), include the real error in the message, close the `HTTPError`'s response with `with`, and shut the server down in `finally`.

</details>

In [ ]:
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_json(404, {"error": "not found"})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)

try:
    with urllib.request.urlopen(base + "/missing", timeout=5) as response:
        print(response.status)
except urllib.error.URLError:
    print("network problem: check your connection")
except urllib.error.HTTPError as err:
    print("server answered", err.code)
server.shutdown()
server.server_close()


### NB30-Q20 · L4 · HTTP requests · Refactor

This client works, but it opens, reads, closes and decodes by hand,
and it would leak the connection if anything in between failed.
Rewrite the request part with `with` and `json.load`.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_json(200, {"pears": 0, "apples": 12, "plums": 7})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)

response = urllib.request.urlopen(base + "/stock", timeout=5)
raw = response.read()
response.close()
text = raw.decode("utf-8")
data = json.loads(text)
for name in sorted(data):
    print(name, data[name])
server.shutdown()
server.server_close()
```

<details><summary><b>Hint</b></summary>

`urlopen` returns a context manager. `json.load` (no `s`) reads straight from a file-like object, and a response is one.

</details>

<details><summary><b>Expected output</b></summary>

```text
apples 12
pears 0
plums 7
```

</details>

<details><summary><b>Solution</b></summary>

```python
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_json(200, {"pears": 0, "apples": 12, "plums": 7})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)

with urllib.request.urlopen(base + "/stock", timeout=5) as response:
    data = json.load(response)
for name in sorted(data):
    print(name, data[name])
server.shutdown()
server.server_close()
```

`with` closes the response however the block ends, including on an exception. `json.load` reads, decodes and parses in one step (JSON is UTF-8 by definition), so three temporary names disappear.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_json(200, {"pears": 0, "apples": 12, "plums": 7})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)

try:
    with urllib.request.urlopen(base + "/stock", timeout=5) as response:
        stock = json.load(response)
finally:
    server.shutdown()
    server.server_close()
for name, qty in sorted(stock.items()):
    print(name, qty)
```

The senior also shuts the server down in `finally` and stops using it as soon as the data has arrived. Holding a connection only as long as needed matters with real connection limits.

</details>

In [ ]:
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_json(200, {"pears": 0, "apples": 12, "plums": 7})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)

response = urllib.request.urlopen(base + "/stock", timeout=5)
raw = response.read()
response.close()
text = raw.decode("utf-8")
data = json.loads(text)
for name in sorted(data):
    print(name, data[name])
server.shutdown()
server.server_close()


### NB30-Q21 · L4 · http.server · Predict

The handler only defines `do_GET`. The client sends a GET, then a
DELETE. Predict the output.

**What does this print?**

```python
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_json(200, {"todos": []})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)

with urllib.request.urlopen(base + "/todos", timeout=5) as response:
    print(response.status)
request = urllib.request.Request(base + "/todos", method="DELETE")
try:
    urllib.request.urlopen(request, timeout=5)
except urllib.error.HTTPError as err:
    print(err.code)
server.shutdown()
server.server_close()
```

<details><summary><b>Hint</b></summary>

`http.server` looks for a method called `do_` plus the verb. What should a server answer when it doesn't support a method at all?

</details>

<details><summary><b>Expected output</b></summary>

```text
200
501
```

</details>

<details><summary><b>Solution</b></summary>

`200`, then `501`. The GET finds `do_GET`. For the DELETE there is no `do_DELETE`, so `http.server` answers `501 Not Implemented` by itself: the server doesn't support that method anywhere. (Compare `405 Method Not Allowed` from Q10, which a REST API sends when the method exists but not for that path.)

</details>

<details><summary><b>Senior dev solution</b></summary>

Frameworks reply `405` with an `Allow` header listing the methods a path accepts, which is more useful to a client. Either way, an unsupported method should get a clear status, never a crash.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## L5 · Hard

### NB30-Q22 · L5 · Retries and backoff · Write

`/flaky` fails with `503` twice, then works. Write
`get_with_retries(url, attempts=4, base_delay=0.01)`: on a **5xx**, print
the attempt, the code and the wait, sleep for `base_delay * 2 **
(attempt - 1)` seconds, and try again; on success, print the attempt,
status and JSON and return the data. A **4xx** must not be retried:
re-raise it. If every attempt fails, raise the last error. Call it on
`/flaky`, then shut the server down.

```text
attempt 1: 503, waiting 0.01s
attempt 2: 503, waiting 0.02s
attempt 3: 200 {'status': 'ok'}
```

**Starting code (already in the cell below):**

```python
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    hits = 0

    def do_GET(self):
        Handler.hits += 1
        if Handler.hits <= 2:
            self.send_json(503, {"error": "busy"})
        else:
            self.send_json(200, {"status": "ok"})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)

import time
```

<details><summary><b>Hint</b></summary>

Loop `for attempt in range(1, attempts + 1)`. Inside, `try` the request; in `except HTTPError as err`, `raise` straight away for codes below 500. On the last attempt, don't sleep: re-raise.

</details>

<details><summary><b>Expected output</b></summary>

```text
attempt 1: 503, waiting 0.01s
attempt 2: 503, waiting 0.02s
attempt 3: 200 {'status': 'ok'}
```

</details>

<details><summary><b>Solution</b></summary>

```python
def get_with_retries(url, attempts=4, base_delay=0.01):
    for attempt in range(1, attempts + 1):
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                data = json.load(response)
                print(f"attempt {attempt}: {response.status} {data}")
                return data
        except urllib.error.HTTPError as err:
            if err.code < 500 or attempt == attempts:
                raise
            wait = base_delay * 2 ** (attempt - 1)
            print(f"attempt {attempt}: {err.code}, waiting {wait}s")
            time.sleep(wait)

get_with_retries(base + "/flaky")
server.shutdown()
server.server_close()
```

Each 503 is retried after a doubling wait (0.01s, then 0.02s), and the third attempt succeeds. The traps: retrying a 4xx (it will never succeed), sleeping after the final attempt (pointless delay), and swallowing the last error (the caller must learn the call failed). A bare `raise` re-raises the current exception with its traceback.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def get_with_retries(url, attempts=4, base_delay=0.01, retry_on=(500, 502, 503, 504)):
    for attempt in range(1, attempts + 1):
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                data = json.load(response)
                print(f"attempt {attempt}: {response.status} {data}")
                return data
        except urllib.error.HTTPError as err:
            err.close()
            if err.code not in retry_on or attempt == attempts:
                raise
            wait = base_delay * 2 ** (attempt - 1)
            print(f"attempt {attempt}: {err.code}, waiting {wait}s")
            time.sleep(wait)

try:
    get_with_retries(base + "/flaky")
finally:
    server.shutdown()
    server.server_close()
```

An explicit `retry_on` set documents which failures are worth retrying and lets callers change it. Retries are only safe for *idempotent* requests (GET, PUT, DELETE): retrying a POST that actually succeeded can create a duplicate order. Libraries like `urllib3.Retry` and `tenacity` package this pattern.

</details>

In [ ]:
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Handler(BaseHTTPRequestHandler):
    hits = 0

    def do_GET(self):
        Handler.hits += 1
        if Handler.hits <= 2:
            self.send_json(503, {"error": "busy"})
        else:
            self.send_json(200, {"status": "ok"})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)

import time

# your code here


### NB30-Q23 · L5 · asyncio streams · Write

Rebuild the echo server with `asyncio` (NB27). Start it with
`asyncio.start_server(handle, "127.0.0.1", 0)`, where `handle` reads one
line and writes it back in capitals. Write `ask(message)` that opens a
connection with `asyncio.open_connection`, sends the message plus
`\n`, and returns the reply without the newline, all inside
`async with asyncio.timeout(2)`. Send the three `messages` concurrently
with `asyncio.gather`, print each reply, then close the server.

```text
PING
ORDER 7 READY
BYE
```

**Starting code (already in the cell below):**

```python
import asyncio

messages = ["ping", "order 7 ready", "bye"]
```

<details><summary><b>Hint</b></summary>

`reader.readline()` and `writer.write(...)` / `await writer.drain()`. Get the address from `server.sockets[0].getsockname()`. Close with `server.close()` then `await server.wait_closed()`; use top-level `await`, never `asyncio.run()`.

</details>

<details><summary><b>Expected output</b></summary>

```text
PING
ORDER 7 READY
BYE
```

</details>

<details><summary><b>Solution</b></summary>

```python
async def handle(reader, writer):
    line = await reader.readline()
    writer.write(line.upper())
    await writer.drain()
    writer.close()
    await writer.wait_closed()

server = await asyncio.start_server(handle, "127.0.0.1", 0)
host, port = server.sockets[0].getsockname()[:2]

async def ask(message):
    async with asyncio.timeout(2):
        reader, writer = await asyncio.open_connection(host, port)
        writer.write(message.encode() + b"\n")
        await writer.drain()
        reply = await reader.readline()
        writer.close()
        await writer.wait_closed()
    return reply.decode().rstrip()

replies = await asyncio.gather(ask(messages[0]), ask(messages[1]), ask(messages[2]))
for reply in replies:
    print(reply)
server.close()
await server.wait_closed()
```

`asyncio.start_server` calls `handle` for each connection with a *reader* and a *writer*: the async version of `recv` and `sendall`. Ending each message with `\n` and using `readline` frames the messages (Q18). `gather` runs the three clients at the same time on one thread and returns the replies in argument order, whichever finished first. `asyncio.timeout` bounds the whole exchange, not each step.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
async def handle(reader, writer):
    async with asyncio.timeout(2):
        line = await reader.readline()
        writer.write(line.upper())
        await writer.drain()
    writer.close()
    await writer.wait_closed()

async def ask(host, port, message):
    async with asyncio.timeout(2):
        reader, writer = await asyncio.open_connection(host, port)
        try:
            writer.write(f"{message}\n".encode())
            await writer.drain()
            return (await reader.readline()).decode().rstrip()
        finally:
            writer.close()
            await writer.wait_closed()

server = await asyncio.start_server(handle, "127.0.0.1", 0)
async with server:
    host, port = server.sockets[0].getsockname()[:2]
    async with asyncio.TaskGroup() as group:
        tasks = [group.create_task(ask(host, port, m)) for m in messages]
for task in tasks:
    print(task.result())
```

`async with server` closes the server however the block exits, and a `TaskGroup` (3.11+) cancels the other clients if one fails, which `gather` does not. The server-side timeout stops one slow client from holding a connection open forever.

</details>

In [ ]:
import asyncio

messages = ["ping", "order 7 ready", "bye"]

# your code here


### NB30-Q24 · L5 · requests and httpx · Write

`requests.get(url, params=..., timeout=...)` builds the query string,
enforces the timeout, and gives you `.raise_for_status()` and `.json()`.
Write the standard-library version: `get_json(path, params=None,
timeout=2)` that adds `params` with `urlencode` when given, returns the
parsed JSON, and turns an `HTTPError` into your own
`ApiError(f"{code} for {path}")`. Then fetch `/search` with
`{"q": "red shoes", "page": 2}`, fetch `/admin` (catch the `ApiError`
and print `error:` with it), and shut the server down.

```text
{'q': 'red shoes', 'page': '2'}
error: 403 for /admin
```

**Starting code (already in the cell below):**

```python
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlencode, urlsplit

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        parts = urlsplit(self.path)
        if parts.path == "/search":
            query = parse_qs(parts.query)
            self.send_json(200, {key: values[0] for key, values in query.items()})
        else:
            self.send_json(403, {"error": "forbidden"})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)
```

<details><summary><b>Hint</b></summary>

Define `class ApiError(Exception): pass`. Inside `get_json`, `try` the request and `raise ApiError(...) from err` in the `except` clause.

</details>

<details><summary><b>Expected output</b></summary>

```text
{'q': 'red shoes', 'page': '2'}
error: 403 for /admin
```

</details>

<details><summary><b>Solution</b></summary>

```python
class ApiError(Exception):
    pass

def get_json(path, params=None, timeout=2):
    url = base + path
    if params:
        url = url + "?" + urlencode(params)
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            return json.load(response)
    except urllib.error.HTTPError as err:
        raise ApiError(f"{err.code} for {path}") from err

print(get_json("/search", {"q": "red shoes", "page": 2}))
try:
    get_json("/admin")
except ApiError as err:
    print("error:", err)
server.shutdown()
server.server_close()
```

This is the core of what `requests` gives you: parameters encoded for you, a timeout on every call, JSON decoding, and errors you choose to raise. The page comes back as `'2'` because everything in a URL is text, a reminder to convert on the server side. `raise ... from err` keeps the original `HTTPError` attached for debugging.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class ApiError(Exception):
    def __init__(self, status, path):
        super().__init__(f"{status} for {path}")
        self.status = status

class ApiClient:
    def __init__(self, base_url, timeout=2):
        self.base_url = base_url
        self.timeout = timeout

    def get_json(self, path, params=None):
        query = f"?{urlencode(params)}" if params else ""
        request = urllib.request.Request(self.base_url + path + query,
                                         headers={"Accept": "application/json"})
        try:
            with urllib.request.urlopen(request, timeout=self.timeout) as response:
                return json.load(response)
        except urllib.error.HTTPError as err:
            err.close()
            raise ApiError(err.code, path) from err

api = ApiClient(base)
try:
    print(api.get_json("/search", {"q": "red shoes", "page": 2}))
    try:
        api.get_json("/admin")
    except ApiError as err:
        print("error:", err)
finally:
    server.shutdown()
    server.server_close()
```

A client class keeps the base URL and timeout in one place and keeps the status code on the exception so callers can branch on it. `requests.Session` and `httpx.Client` work the same way and add connection reuse (much faster for many calls), cookies, retries via adapters and, in `httpx`, async support and HTTP/2.

</details>

In [ ]:
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlencode, urlsplit

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        parts = urlsplit(self.path)
        if parts.path == "/search":
            query = parse_qs(parts.query)
            self.send_json(200, {key: values[0] for key, values in query.items()})
        else:
            self.send_json(403, {"error": "forbidden"})

    def send_json(self, status, data):
        body = json.dumps(data).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook quiet

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

server, base = start_server(Handler)

# your code here


---

## Mini-project

Pull together everything from this notebook.

## NB30-P · L5 · Mini-project: Todo REST Service · Write

Build a tiny REST service for todos and a client that exercises it.

**Server.** A `Handler` (with the given `send_json` and `log_message`)
that stores todos in the class-level dict `todos` and numbers them
from `1` using `next_id`:

- `GET /todos`: `200` and the list of todos ordered by id.
- `POST /todos` with JSON `{"title": ...}`: `201` and the new todo
  `{"id": ..., "title": ..., "done": false}`. A missing or empty title
  gives `400` and `{"error": "title is required"}`.
- `DELETE /todos/<id>`: `204` with no body, or `404` and
  `{"error": "not found"}` if there's no such todo.
- Anything else: `404` and `{"error": "not found"}`.

**Client.** Write `call(method, path, payload=None)` that sends the
request (JSON body when `payload` is given, a 5-second timeout),
returns `(status, data)` for **both** successes and `HTTPError`s, and
uses `None` as the data when the body is empty. Run this script and
print one line per call as `METHOD PATH -> STATUS DATA`, then shut the
server down:

```text
POST /todos -> 201 {'id': 1, 'title': 'Buy milk', 'done': False}
POST /todos -> 201 {'id': 2, 'title': 'Write report', 'done': False}
POST /todos -> 400 {'error': 'title is required'}
GET /todos -> 200 [{'id': 1, 'title': 'Buy milk', 'done': False}, {'id': 2, 'title': 'Write report', 'done': False}]
DELETE /todos/1 -> 204 None
DELETE /todos/1 -> 404 {'error': 'not found'}
GET /todos -> 200 [{'id': 2, 'title': 'Write report', 'done': False}]
```

The calls, in order: POST `Buy milk`, POST `Write report`, POST
`{"title": ""}`, GET, DELETE `/todos/1` twice, GET.

**Starting code (already in the cell below):**

```python
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

# Paste these two methods into your Handler class:
def send_json(self, status, data):
    body = json.dumps(data).encode()
    self.send_response(status)
    self.send_header("Content-Type", "application/json")
    self.send_header("Content-Length", str(len(body)))
    self.end_headers()
    self.wfile.write(body)

def log_message(self, format, *args):
    pass
```

<details><summary><b>Hint</b></summary>

For DELETE, take the id with `self.path.removeprefix("/todos/")` and check `isdigit()` before `int()`. A `204` is `send_response(204)` plus `end_headers()`, no body. In `call`, read the body with `.read()` in both the success and the `HTTPError` branch (an `HTTPError` can be read like a response) and `json.loads` it only if it isn't empty.

</details>

<details><summary><b>Expected output</b></summary>

```text
POST /todos -> 201 {'id': 1, 'title': 'Buy milk', 'done': False}
POST /todos -> 201 {'id': 2, 'title': 'Write report', 'done': False}
POST /todos -> 400 {'error': 'title is required'}
GET /todos -> 200 [{'id': 1, 'title': 'Buy milk', 'done': False}, {'id': 2, 'title': 'Write report', 'done': False}]
DELETE /todos/1 -> 204 None
DELETE /todos/1 -> 404 {'error': 'not found'}
GET /todos -> 200 [{'id': 2, 'title': 'Write report', 'done': False}]
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Handler(BaseHTTPRequestHandler):
    todos = {}
    next_id = 1

    def do_GET(self):
        if self.path == "/todos":
            items = []
            for todo_id in sorted(Handler.todos):
                items.append(Handler.todos[todo_id])
            self.send_json(200, items)
        else:
            self.send_json(404, {"error": "not found"})

    def do_POST(self):
        if self.path != "/todos":
            self.send_json(404, {"error": "not found"})
            return
        length = int(self.headers.get("Content-Length", 0))
        data = json.loads(self.rfile.read(length) or b"{}")
        title = data.get("title", "")
        if not title:
            self.send_json(400, {"error": "title is required"})
            return
        todo = {"id": Handler.next_id, "title": title, "done": False}
        Handler.todos[Handler.next_id] = todo
        Handler.next_id += 1
        self.send_json(201, todo)

    def do_DELETE(self):
        text_id = self.path.removeprefix("/todos/")
        if self.path.startswith("/todos/") and text_id.isdigit() and int(text_id) in Handler.todos:
            del Handler.todos[int(text_id)]
            self.send_response(204)
            self.end_headers()
        else:
            self.send_json(404, {"error": "not found"})

    send_json = send_json
    log_message = log_message

server, base = start_server(Handler)

def call(method, path, payload=None):
    body = None
    headers = {}
    if payload is not None:
        body = json.dumps(payload).encode()
        headers["Content-Type"] = "application/json"
    request = urllib.request.Request(base + path, data=body, method=method,
                                     headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=5) as response:
            status = response.status
            raw = response.read()
    except urllib.error.HTTPError as err:
        status = err.code
        raw = err.read()
        err.close()
    data = json.loads(raw) if raw else None
    return status, data

script = [
    ("POST", "/todos", {"title": "Buy milk"}),
    ("POST", "/todos", {"title": "Write report"}),
    ("POST", "/todos", {"title": ""}),
    ("GET", "/todos", None),
    ("DELETE", "/todos/1", None),
    ("DELETE", "/todos/1", None),
    ("GET", "/todos", None),
]
for method, path, payload in script:
    status, data = call(method, path, payload)
    print(f"{method} {path} -> {status} {data}")
server.shutdown()
server.server_close()
```

The server keeps state in class attributes because `http.server` creates a **new** handler object for every request; instance attributes would vanish between calls. Each route returns the conventional status: `201` with the created resource, `204` with no body, `400` for bad input, `404` for a missing item. The client treats an `HTTPError` as an ordinary answer (it *is* a response, with a status and a body), and only the empty `204` body becomes `None`. The second DELETE shows why REST clients must handle `404` gracefully.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import re
import threading as _threading
from http import HTTPStatus

TODO_PATH = re.compile(r"/todos/(\d+)")

class TodoHandler(BaseHTTPRequestHandler):
    todos = {}
    next_id = 1
    lock = _threading.Lock()

    def do_GET(self):
        if self.path != "/todos":
            return self.not_found()
        with self.lock:
            items = [self.todos[k] for k in sorted(self.todos)]
        self.send_json(HTTPStatus.OK, items)

    def do_POST(self):
        if self.path != "/todos":
            return self.not_found()
        length = int(self.headers.get("Content-Length", 0))
        try:
            title = json.loads(self.rfile.read(length) or b"{}").get("title")
        except (json.JSONDecodeError, AttributeError):
            title = None
        if not isinstance(title, str) or not title.strip():
            return self.send_json(HTTPStatus.BAD_REQUEST, {"error": "title is required"})
        with self.lock:
            cls = type(self)
            todo = {"id": cls.next_id, "title": title, "done": False}
            cls.todos[cls.next_id] = todo
            cls.next_id += 1
        self.send_json(HTTPStatus.CREATED, todo)

    def do_DELETE(self):
        match = TODO_PATH.fullmatch(self.path)
        with self.lock:
            removed = match and self.todos.pop(int(match[1]), None)
        if not removed:
            return self.not_found()
        self.send_response(HTTPStatus.NO_CONTENT)
        self.end_headers()

    def not_found(self):
        self.send_json(HTTPStatus.NOT_FOUND, {"error": "not found"})

    send_json = send_json
    log_message = log_message

def call(base_url, method, path, payload=None, timeout=5):
    body = None if payload is None else json.dumps(payload).encode()
    headers = {"Content-Type": "application/json"} if body else {}
    request = urllib.request.Request(base_url + path, data=body, method=method, headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return response.status, json.loads(response.read() or b"null")
    except urllib.error.HTTPError as err:
        with err:
            return err.code, json.loads(err.read() or b"null")

script = [
    ("POST", "/todos", {"title": "Buy milk"}),
    ("POST", "/todos", {"title": "Write report"}),
    ("POST", "/todos", {"title": ""}),
    ("GET", "/todos", None),
    ("DELETE", "/todos/1", None),
    ("DELETE", "/todos/1", None),
    ("GET", "/todos", None),
]
server, base = start_server(TodoHandler)
try:
    for method, path, payload in script:
        status, data = call(base, method, path, payload)
        print(f"{method} {path} -> {status} {data}")
finally:
    server.shutdown()
    server.server_close()
```

`ThreadingHTTPServer` handles each request on its own thread, so the senior guards the shared dict with a `Lock` (NB27). A regex (NB17) with `fullmatch` parses `/todos/<id>` strictly, `HTTPStatus` names replace magic numbers, bad JSON becomes a `400` instead of a crash, and `finally` guarantees shutdown. A framework such as FastAPI gives you routing, validation and docs for free, but the HTTP underneath is exactly this.

</details>

In [ ]:
import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

def start_server(handler_class):
    server = ThreadingHTTPServer(("127.0.0.1", 0), handler_class)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"

# Paste these two methods into your Handler class:
def send_json(self, status, data):
    body = json.dumps(data).encode()
    self.send_response(status)
    self.send_header("Content-Type", "application/json")
    self.send_header("Content-Length", str(len(body)))
    self.end_headers()
    self.wfile.write(body)

def log_message(self, format, *args):
    pass

# your code here


---

## Done

Next up: **NB31 · Real World Programs** in `advanced/`.